In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_pinball_loss
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
import seaborn as sns
import torch
from torch import nn
import torch.optim as optim
from collections import defaultdict

np.random.seed(42)


In [ ]:
# Load the data
scores = pd.read_parquet('Scores.parquet', engine='fastparquet')
qualities = pd.read_parquet('Qualities.parquet', engine='fastparquet')

In [ ]:
# Merge the scores and qualities dataframes
df = scores[scores["diag"] == 1] #! Genuine (1) or Imposter (0)
df = df.merge(qualities.add_suffix("_1"), left_on="image_1", right_on="image_1", how="inner")
df = df.merge(qualities.add_suffix("_2"), left_on="image_2", right_on="image_2", how="inner")

In [ ]:
df.head()

In [ ]:
# Define the pinball loss function for quantile regression
def pinball_loss(y_true, y_pred, quantile):
    u = y_true - y_pred
    loss = torch.mean(torch.max(quantile * u, (quantile - 1) * u))
    return loss

In [ ]:
# Implement the phi transformation for symmetric features
def make_sym_features(df, base_cols):
        out = {}
        for col in base_cols:
            col1 = f"{col}_1"
            col2 = f"{col}_2"
            out[f"{col}_sum"] = df[col1] + df[col2]
            out[f"{col}_diff"] = np.abs(df[col1] - df[col2])
        return pd.DataFrame(out)

In [ ]:
# Prepare the data for training
X = df.drop(columns=["diag", "image_1", "image_2", "score"])
features = qualities.columns.tolist()# Base feature names without suffixes
features.remove("image") 
X = make_sym_features(X, features)
y = df["score"].values
list_features = X.columns.tolist()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
del X, y, df
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
quantiles = [0.01] + [round(x, 2) for x in np.linspace(0, 1, 21)[1:-1]] + [0.99] #Quantiles to evaluate

In [ ]:
# Train a NN for quantile regression
class NNQuantileRegressor(nn.Module):
    def __init__(self, input_dim, hidden_units, quantile):
        super(NNQuantileRegressor, self).__init__()
        self.quantile = quantile
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze()

def train_nn(X_train, y_train, X_val, y_val, quantile=0.5, epochs=100, batch_size=64, learning_rate=0.001):
    input_dim = X_train.shape[1]
    model = NNQuantileRegressor(input_dim, hidden_units=64, quantile=quantile)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = lambda y_true, y_pred: pinball_loss(y_true, y_pred, quantile)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

    train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_batch, y_pred)
            loss.backward()
            optimizer.step()
        if (epoch+1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_tensor)
                val_loss = criterion(y_val_tensor, val_pred)
            print(f"Epoch {epoch+1}/{epochs}, Validation Pinball Loss: {val_loss.item():.4f}")
    return model


In [ ]:
# Run the training for each quantile
models_nn = {}
for q in quantiles:
    print(f"Training model for quantile {q}")
    model_nn_q = train_nn(X_train, y_train, X_val, y_val, quantile=q)
    models_nn[q] = model_nn_q

In [ ]:
# Evalute the models
def evaluate_model(model, X_val, y_val, quantile=0.5):
    with torch.no_grad():
        y_pred = model(torch.tensor(X_val, dtype=torch.float32)).numpy()
    loss = mean_pinball_loss(y_val, y_pred, alpha=quantile)
    # Constant quantile predictor
    q_constant = np.quantile(y_val, quantile)
    y_pred_constant = np.full_like(y_val, fill_value=q_constant)
    loss_constant = mean_pinball_loss(y_val, y_pred_constant, alpha=quantile)
    s = 1- loss/loss_constant
    return loss, loss_constant, s

loss_results = {}
for q in quantiles:
    #print(f"Evaluating model for quantile {q}")
    model_q = models_nn[q]
    loss_results[q] = evaluate_model(model_q, X_val, y_val, quantile=q)

# Plot the results
plt.figure(figsize=(10, 6))
quantiles = sorted(loss_results.keys())
losses = [loss_results[q][2] for q in quantiles]
plt.plot(quantiles, losses, marker='o')
plt.title('Relative Improvement vs Quantiles')
plt.xlabel('Quantile (τ)')
plt.ylabel('Relative Improvement')
plt.grid()
plt.show()

In [ ]:
class TorchModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)
        return self.model(x)

shap_values_dict = {}

for q in quantiles:
    print(f"Computing SHAP values for quantile {q}")

    model_q = models_nn[q].eval()
    wrapped_model = TorchModelWrapper(model_q)

    explainer = shap.Explainer(wrapped_model, X_train)

    shap_values_q = explainer(
        X_val,
        check_additivity=False
    )

    shap_values_dict[q] = shap_values_q

In [ ]:
def beeswarm(shap_values_dict, max_display = 20):
    #for quantile in shap_values_dict.keys(): #for all quantiles
    for quantile in [0.1, 0.5, 0.9]: #for selected quantiles
        shap_values_dict[quantile].feature_names = list_features
        plt.figure(figsize=(10, 8))
        shap.plots.beeswarm(shap_values_dict[quantile], max_display=max_display, show=False, group_remaining_features = False)
        plt.title('Most Important Variables')
        #plt.savefig(f'img/Quantile-{str(quantile)}-Beeswarm.pdf',  bbox_inches='tight')
        plt.show()
    return
beeswarm(shap_values_dict, max_display=6) #selected_features = most_important_features)

In [ ]:
def feature_importance(shap_values_dict):
    cols = shap_values_dict[0.5].feature_names
    shap_importance_list = []
    for quantile in shap_values_dict.keys():
        shap_values = shap_values_dict[quantile]
        shap_importance = np.abs(shap_values.values).mean(axis=0)
        shap_importance_list.append(np.array(shap_importance))
    shap_importance_list = np.array(shap_importance_list).T
    df = pd.DataFrame(shap_importance_list, cols, shap_values_dict.keys())
    return df

def plot_feature_importance(df, start_end = (0, 10), features_set = None):
    sorted_features = sorted(zip(list(df.index), df[0.5]), key = lambda x: -x[1])
    most_important_features = [sorted_features[i][0] for i in range(start_end[0], start_end[1])]
    if features_set is not None:
        most_important_features = features_set
    plt.figure(figsize=(8,6))
    sns.heatmap(df.loc[most_important_features], annot=False, cmap="YlGnBu")  # annot=True shows numbers
    plt.title("Heatmap of DataFrame")
    plt.xlabel("Quantiles")
    plt.ylabel("Features")
    #plt.savefig(f'img/Feature_Importance_Heatmap_{start_end[0]}_{start_end[1]}.pdf',  bbox_inches='tight')
    plt.show()
    return most_important_features

df = feature_importance(shap_values_dict)
important_features = defaultdict(float)
for quantile in shap_values_dict.keys():
    sorted_features = sorted(zip(list(df.index), df[quantile]), key = lambda x: -x[1])
    most_important_features = [{sorted_features[i][0]: sorted_features[i][1]} for i in range(0, 5)]
    for feature, value in sorted_features[:5]:
        important_features[feature] += value
relevant_features = sorted(important_features.items(), key=lambda x: -x[1])
relevant_features = [feat for feat, val in relevant_features]
most_important_features = plot_feature_importance(df, start_end=(0, 40), features_set=relevant_features)


In [ ]:
# -- SHAP Impact Across Predicted Score --
def shap_impact_across_score(
    model, shap_values, X, feature_names, feature_of_interest, nb_bins=20
):
    """Average SHAP value of a feature across predicted score bins."""
    
    feat_idx = feature_names.index(feature_of_interest)
    with torch.no_grad():
        predicted_scores = model(torch.tensor(X, dtype=torch.float32)).numpy()
    df = pd.DataFrame({
        "predicted_score": predicted_scores,
        "shap_value": shap_values.values[:, feat_idx],
    })

    return (
        df.assign(score_bin=pd.qcut(df.predicted_score, nb_bins, duplicates="drop"))
          .groupby("score_bin", as_index=False)
          .agg(
              mean_shap=("shap_value", "mean"),
              score_center=("predicted_score", "mean"),
          )
    )
def plot_shap_feature_across_scores(models, shap_values_dict, X, feature_names, feature_of_interest, quantiles=(0.1, 0.5, 0.9), nb_bins=20):
    plt.figure(figsize=(10, 6))

    for q in quantiles:
        agg = shap_impact_across_score(
            models[q],
            shap_values_dict[q],
            X,
            feature_names,
            feature_of_interest,
            nb_bins,
        )
        plt.plot(agg.score_center, agg.mean_shap, label=f"Quantile {q}", alpha=0.8)

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Predicted score")
    plt.ylabel(f"SHAP value of {feature_of_interest}")
    plt.title(f"Impact of {feature_of_interest} across score levels")
    plt.legend()
    plt.grid(True)
    #plt.savefig(f'img/SHAP_Impact_{feature_of_interest}.pdf')
    plt.show()
for feature in most_important_features[:5]:
    plot_shap_feature_across_scores(
        models=models_nn,
        shap_values_dict=shap_values_dict,
        X=X_val,
        feature_names=list_features,
        feature_of_interest=feature,
        quantiles=[0.1, 0.5, 0.9], #[0.05, 0.3, 0.5, 0.7, 0.95],
        nb_bins=50
    )

In [ ]:
def plot_one_feature(X, models, features_name, feature_of_interest, nb_bins = 100):
    plt.figure(figsize=(10, 6))
    for quantile, model in models.items():
        with torch.no_grad():
            y_pred = model(torch.tensor(X, dtype=torch.float32)).numpy()
        df = pd.DataFrame(X.T[features_name.index(feature_of_interest)], columns=[feature_of_interest])
        df['predicted_score'] = y_pred
        df['bin'] = pd.cut(df[feature_of_interest], bins=nb_bins)
        bin_avg = df.groupby('bin')['predicted_score'].mean().reset_index()
        bin_avg['bin_center'] = bin_avg['bin'].apply(lambda x: x.mid)
        plt.plot(bin_avg['bin_center'], bin_avg['predicted_score'], alpha=0.5, label=f'Quantile {quantile}')
        #plt.scatter(df[feature_of_interest], df['predicted_score'], alpha=0.1, label=f'Quantile {quantile}')
    plt.title(f"{feature_of_interest} vs Predicted Score")
    plt.xlabel(feature_of_interest)
    plt.ylabel("Predicted Score")
    plt.legend()
    #plt.savefig(f'img/Feature_vs_Predicted_Score_{feature_of_interest}.pdf')
    plt.show()
def quantile_coverage(X, y_true, models , quantiles):
    coverage = []
    for q in quantiles:
        with torch.no_grad():
            y_pred = models[q](torch.tensor(X, dtype=torch.float32)).numpy()
        residual = y_true - y_pred
        coverage.append(np.mean(residual <= 0))
    plt.figure(figsize=(10, 6))
    plt.plot(quantiles, coverage, marker='o', label='Empirical Coverage')
    plt.plot(quantiles, quantiles, linestyle='--', label='Ideal (y = x)')
    plt.title('Quantile Coverage')
    plt.xlabel('Quantile (τ)')
    plt.ylabel('Empirical Coverage')
    plt.grid()
    plt.legend()
    #plt.savefig(f'img/Quantile_Coverage.pdf')
    plt.show()
feature = "age_1_diff"
plot_one_feature(X_train, {q: models_nn[q] for q in [0.1, 0.5, 0.9]}, list_features, feature, nb_bins = 20)
quantile_coverage(X_train, y_train, models_nn , quantiles)